## Практика: полная предобработка данных

Ваша цель — подготовить данные для линейной регрессии с учётом всех ключевых этапов предобработки:
- обработка пропусков;
- обработка выбросов;
- обработка категориальных признаков;
- масштабирование числовых признаков;
- создание единой структуры предобработки с помощью ColumnTransformer.

Когда вы выполните все эти шаги, у вас получится трансформер, который можно использовать совместно с моделью линейной регрессии.

## Подключение и загрузка бибилотек

In [1]:
import numpy as np
import pandas as pd


# Библиотеки машинного обучения
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
## Перед отправкой ревьюверу - поменять путь
df = pd.read_csv("../data/turtles.csv", sep='\t', decimal=',')

## Убираем лишние колонки
df = df.drop(['id', 'registration number', 'timestamp'], axis=1)

### Ожидаемые данные

- `id` — идентификатор измерения.
- `binomial_name` — международное научное название вида черепахи.
- `registration_number` — регистрационный номер черепахи.
- `shell_length` — длина панциря, мм.
- `shell_width` — ширина панциря, мм.
- `head_length` — длина головы, мм.
- `head_width` — ширина головы, мм.
- `flipper_length_n` — длина одной ласты, мм. У черепах четыре ласты, поэтому в датасете четыре таких столбца, в названиях вместо n указан номер от 1 до 4.
- `flipper_width_n` — ширина одной ласты, мм. У черепах четыре ласты, поэтому в датасете четыре таких столбца, в названиях вместо n указан номер от 1 до 4.
- `circle_count` — количество колец роста на панцире черепахи.
- `measure_count` — количество измерений, которые произвели, прежде чем усреднить показатели черепахи и добавить их в таблицу.
- `shell_crack` — наличие трещин панциря.
- `timestamp` — время внесения данных о черепахе.
- `weight` — масса черепахи, кг.

## Задача 1. Обработка пропусков

Напомним, как формируются наши данные о черепахах: их размеры фиксируются системой компьютерного зрения "TurtleCV", которая проводит измерения, пока черепахи находятся "на виду". Система работает неидеально, и иногда ей не удаётся определить некоторые параметры.

In [3]:
## Выведем значения пропусков
empty = df.isna().mean() * 100
empty.loc[empty > 0]

binomial_name        0.552985
shell_length         0.981830
head_length          1.647670
head_width           1.647670
flipper_length_3     1.139826
flipper_width_3      1.139826
flipper_length_4     1.139826
flipper_width_4      1.139826
measure_count        2.979348
shell_crack         75.442952
weight               0.214423
dtype: float64

Для обработки пропусков нужно использовать несколько стратегий.

- Для числовых признаков, которые описывают размеры (длина/ширина панциря, головы, ласт и т.д.), будем использовать медиану. Биометрические параметры часто имеют ассиметричное распределение и выбросы (подростки vs взрослые особи, редкие крупные экземпляры). В отличие от среднего, медиана устойчива к подобным выбросам и даёт более стабильные значения.
- `binomial_name` — это категориальный признак (вид черепахи). Для категорий нельзя посчитать среднее или медиану, и наиболее подходящая замена — самое частое значение признака, или мода. Использование моды сохраняет реалистичность данных и не создаёт фиктивных категорий.
- Особый случай — признак `shell_crack` (количество трещин на панцире). Здесь пропуск, скорее всего, означает отсутствие трещин, поэтому логично заполнить его нулём. Аналогично поступим с признаком weight (вес), чтобы впоследствии обработать аномальные значения.

### Задание 1.1
Для каждой группы признаков создайте отдельный объект SimpleImputer:
- для числовых признаков размеров — с параметром strategy="median";
- для категориальных признаков — с параметром strategy="most_frequent";
- для признака shell_crack — с параметрами strategy="constant" и fill_value=0.

In [4]:

# Списки признаков
cat_features = ['binomial_name']
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count','weight'
]
special_features = ['shell_crack']

# Создайте объекты SimpleImputer для каждой группы признаков:
cat_imputer = SimpleImputer(strategy="most_frequent")
num_imputer = SimpleImputer(strategy="median")
zero_imputer = SimpleImputer(strategy="constant", fill_value=0)

# Объедините всё в ColumnTransformer и примените трансформер к данным:
imputer_ct = ColumnTransformer(
    transformers=[
        ('cat_frequent', cat_imputer, cat_features),
        ('num_mean', num_imputer, num_features),
        ('num_zero', zero_imputer, special_features),
    ],
    remainder='passthrough', # Оставляем не только трасформационные, но и другие ячейки
    verbose_feature_names_out=False # не меняем название колонок
)

# При трансформации - 
# Чтобы не мучиться с именами колонок вручную:
imputer_ct.set_output(transform="pandas")

# 2. Превращаем обратно в DataFrame, сохраняя порядок колонок
df_filled = imputer_ct.fit_transform(df)

print(df_filled.isna().sum())

binomial_name       0
shell_length        0
shell_width         0
head_length         0
head_width          0
flipper_length_1    0
flipper_width_1     0
flipper_length_2    0
flipper_width_2     0
flipper_length_3    0
flipper_width_3     0
flipper_length_4    0
flipper_width_4     0
circle_count        0
measure_count       0
weight              0
shell_crack         0
dtype: int64


## Задача 2. Обработка выбросов


### Удаление строк с некорректным весом

Удалите из датасета строки, где `weight = 0`. Чтобы в дальнейшем не было проблем с обработкой значений, восстановите индексы с помощью `reset_index`. 

Выведите размер датасета до и после удаления строк.



In [5]:
# Удалите строки с некорректным весом 
# Выведите размер датасета до и после удаления
before_shape = df.shape[0]
df = df[df['weight'] > 0].reset_index(drop=True)
after_shape = df.shape[0]
print('before:', before_shape, 'after:', after_shape)

before: 8861 after: 8834


### Исправление шасштаба

В задании указано, что некоторые размеры были случайно умножены на 10

Чтобы не писать длинную последовательность проверок, оформим исправление в отдельную функцию. Для этого напишите функцию fix_scale(series, threshold), которая принимает на вход значения признака и порог. 
Функция должна делить на 10 все значения series, если они больше порога ( threshold ). Так вы исправите случаи, где данные ошибочно умножены на 10, и при этом сохраните редкие большие, но реалистичные значения.

In [6]:
# Реализуйте функцию исправления масштаба
def fix_scale(series, threshold):
    # Используем np.where(условие, значение_если_True, значение_если_False)
    # Если значение больше порога, делим на 10, иначе оставляем как есть
    return np.where(series > threshold, series / 10, series)

# Проверим функцию fix_scale на примере:
test_series = np.array([500, 2600, 10000, 200])
result = fix_scale(test_series, 2500)
print(result)

[ 500.  260. 1000.  200.]


Задание 2.3

Примените написанную функцию ко всем соответствующим столбцам.

Используйте следующие пороги:
- для панциря ( shell_length, shell_width ) — 2500 мм;
- для головы ( head_length, head_width ) — 400 мм;
- для ласт ( flipper_length_*, flipper_width_* ) — 1000 мм.

In [7]:
# Допишите списки признаков для каждой группы
shell_cols = ['shell_length', 'shell_width']
head_cols = ['head_length', 'head_width']
flipper_cols = [   
     'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',]

# Примените функцию к каждой группе признаков
df[shell_cols] = fix_scale(df[shell_cols], 2500)
df[head_cols] = fix_scale(df[head_cols], 400)
df[flipper_cols] = fix_scale(df[flipper_cols], 1000)

# Проверим минимальные и максимальные значения после обработки
cols_to_check = shell_cols + head_cols + flipper_cols
stats = df[cols_to_check].agg(['min', 'max']).T
print(stats)

                    min     max
shell_length      132.0  2494.0
shell_width        78.0  1554.0
head_length        19.0   400.0
head_width         12.0   382.0
flipper_length_1   60.0   997.0
flipper_width_1    34.0  1000.0
flipper_length_2   57.0  1000.0
flipper_width_2    33.0   997.0
flipper_length_3   50.0   997.0
flipper_width_3    27.0   997.0
flipper_length_4   44.0  1000.0
flipper_width_4    25.0   999.0


## Задача 3. Обработка категориальных значений


Нужно закодировать категории черепах

Стратегия кодирования будет выбираться до построения общего трансформера в зависимости от признака:
если уникальных значений `binomial_name` < 10, нужно использовать OneHotEncoder;
если уникальных значений > 10 — TargetEncoder.

### Нормализация регистра и подсчёт уникальных значений


Задание 3.1

В столбце `binomial_name` встречаются записи и в верхнем, и в нижнем регистре. Приведите все значения к нижнему регистру. Выведите количество уникальных категорий до и после нормализации регистра (используйте `nunique()` ).


In [8]:
# Посчитайте число уникальных значений до и после приведения к нижнему регистру
uniq_before = df['binomial_name'].nunique()
df['binomial_name'] = df['binomial_name'].str.lower() 
uniq_after = df['binomial_name'].nunique()
print('unique before:', uniq_before, 'unique after:', uniq_after)

unique before: 23 unique after: 6


Задание 3.2

Выберите метод кодирования в зависимости от полученного количества уникальных значений признака binomial_name:

- до 10 — OneHotEncoder;
- более 10 — TargetEncoder.

Закодируйте `binomial_name` выбранным методом. Получите новый датасет `df_encoded` без исходного столбца `binomial_name`, но с новыми закодированными столбцами. Выведите список столбцов полученного датасета.

In [9]:
# Выберите и обучите кодировщик в зависимости от числа уникальных значений
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# Используем fit_transform для обучения и трансформации за один шаг
encoded_columns = encoder.fit_transform(df[['binomial_name']])

# Соберите итоговый датасет df_encoded:
df_encoded = pd.DataFrame(
    encoded_columns,
    columns=encoder.get_feature_names_out(['binomial_name']) # Получили все новые названия колонок
)
print(list(df_encoded.columns))

['binomial_name_caretta caretta', 'binomial_name_chelonia mydas', 'binomial_name_dermochelys coriacea', 'binomial_name_eretmochelys imbricata', 'binomial_name_lepidochelys kempii', 'binomial_name_lepidochelys olivacea', 'binomial_name_nan']


Задание 4

Создайте объект `StandardScaler` для масштабирования числовых значений.

In [10]:
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count'
]

# Создайте объект StandardScaler
scaler = StandardScaler()

# Примените scaler к датасету
df[num_features] = scaler.fit_transform(df[num_features])

print(df.columns)

Index(['binomial_name', 'shell_length', 'shell_width', 'head_length',
       'head_width', 'flipper_length_1', 'flipper_width_1', 'flipper_length_2',
       'flipper_width_2', 'flipper_length_3', 'flipper_width_3',
       'flipper_length_4', 'flipper_width_4', 'circle_count', 'measure_count',
       'shell_crack', 'weight'],
      dtype='str')


## Задача 5. Создание трансформера


План обработки признаков:

Категориальные признаки — пайплайн cat_pipeline:
- SimpleImputer(…);
- OneHotEncoder(…).

Числовые признаки — пайплайн num_pipeline:
- SimpleImputer(…);
- StandardScaler().

Чтобы объединить пайплайны в единый блок и распределить их по нужным наборам столбцов, используйте ColumnTransformer. Он будет состоять из трёх частей:
- cat_pipeline для категориальных признаков ( cat_features );
- num_pipeline для числовых признаков ( num_features );
- SimpleImputer для особых признаков ( special_features ).

Назовите итоговый трансформер preprocessor. Соберите обработанные признаки в новый датасет df_transformed.

In [16]:
num_features = [
    'shell_length', 'shell_width', 'head_length', 'head_width',
    'flipper_length_1', 'flipper_width_1',
    'flipper_length_2', 'flipper_width_2',
    'flipper_length_3', 'flipper_width_3',
    'flipper_length_4', 'flipper_width_4',
    'circle_count', 'measure_count'
]
cat_features = ['binomial_name']
special_features = ['shell_crack', 'weight']

## SimpleImputer для всех полей
cat_imputer = SimpleImputer(strategy="most_frequent")
num_imputer = SimpleImputer(strategy="median")
zero_imputer = SimpleImputer(strategy="constant", fill_value=0)

# num_scaler
scaler = StandardScaler()

# cat_encoder
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Создайте пайплайны для категориальных и числовых признаков
cat_pipeline = Pipeline(
    steps=[
        ('cat_imputer', cat_imputer),
        ('cat_encoder', cat_encoder),
    ]
)

num_pipeline = Pipeline(
    steps=[
        ('num_imputer', num_imputer),
        ('scaler', scaler)
    ]
)

# Объедините все шаги в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_pipeline', cat_pipeline, cat_features),
        ('num_pipeline', num_pipeline, num_features),
        ('special_features', zero_imputer, special_features),
    ]
)

# Примените транфсормер к датасету
transformed_features = preprocessor.fit_transform(df)
feature_names = preprocessor.get_feature_names_out()

df_transformed = pd.DataFrame(transformed_features, columns=feature_names)
print(df_transformed.head(5))

   cat_pipeline__binomial_name_caretta caretta  \
0                                          1.0   
1                                          0.0   
2                                          0.0   
3                                          0.0   
4                                          0.0   

   cat_pipeline__binomial_name_chelonia mydas  \
0                                         0.0   
1                                         0.0   
2                                         0.0   
3                                         0.0   
4                                         0.0   

   cat_pipeline__binomial_name_dermochelys coriacea  \
0                                               0.0   
1                                               0.0   
2                                               0.0   
3                                               0.0   
4                                               0.0   

   cat_pipeline__binomial_name_eretmochelys imbricata  \
0               